# BI-Safe: Feature Engineering and Isolation Forest Training
This notebook mirrors `ml_pipeline/train_model.py`, `calibrate.py` and `evaluate.py` step by step, so every stage of Section 5.4.1 can be shown with real output.

**Pipeline:** load CICIDS2017 → clean column names → derive 20 features → label (`is_attack`) → remove NaN/Inf → scale → train 4 Isolation Forests → evaluate on unseen attack data.

> Models are saved to `models_notebook/` so the production models in `models/` are **not** overwritten.

## 1. Setup

In [ ]:
import glob, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import dump
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings("ignore")

# Works whether the notebook sits in the project root or in a notebooks/ folder
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "real"
MODEL_DIR = PROJECT_ROOT / "models_notebook"      # separate from production models/
MODEL_DIR.mkdir(exist_ok=True)

SAMPLE_FRAC = 1.0      # use 0.2 for a quick run; 1.0 = full dataset
RANDOM_STATE = 42

# Raw CICIDS column -> normalised feature name (same mapping as train_model.py)
RAW_MAP = {
    "Destination Port":              "destination_port",
    "Flow Duration":                 "duration",
    "Total Length of Fwd Packets":   "bytes_sent",
    "Total Length of Bwd Packets":   "bytes_received",
    "Total Fwd Packets":             "packets_sent",
    "Total Backward Packets":        "packets_received",
    "Packet Length Mean":            "packet_size",
    "Flow Bytes/s":                  "flow_bytes_per_s",
    "Flow Packets/s":                "flow_packets_per_s",
    "SYN Flag Count":                "syn_flag_count",
    "ACK Flag Count":                "ack_flag_count",
    "PSH Flag Count":                "psh_flag_count",
    "Fwd Packet Length Std":         "fwd_packet_len_std",
    "Bwd Packet Length Std":         "bwd_packet_len_std",
    "Average Packet Size":           "avg_packet_size",
    "Down/Up Ratio":                 "down_up_ratio",
    "Init_Win_bytes_forward":        "init_win_bytes_fwd",
    "Flow IAT Mean":                 "flow_iat_mean",
    "Flow IAT Std":                  "flow_iat_std",
}

FEATURE_COLS = [
    "source_port", "destination_port", "duration",
    "bytes_sent", "bytes_received", "packets_sent",
    "packets_received", "packet_size",
    "flow_bytes_per_s", "flow_packets_per_s",
    "syn_flag_count", "ack_flag_count", "psh_flag_count",
    "fwd_packet_len_std", "bwd_packet_len_std",
    "avg_packet_size", "down_up_ratio",
    "init_win_bytes_fwd", "flow_iat_mean", "flow_iat_std",
]
CONTAMINATIONS = [0.01, 0.02, 0.05, 0.10]

print("Project root:", PROJECT_ROOT)
print("Number of selected features:", len(FEATURE_COLS))

## 2. Load the raw CICIDS2017 data and clean the column names
CICIDS column names contain spaces (e.g. `' Destination Port'`), so they are stripped first.

In [ ]:
files = sorted(glob.glob(str(RAW_DIR / "*.csv")))
print(f"Found {len(files)} raw CSV file(s) in {RAW_DIR}")

frames = []
for f in files:
    d = pd.read_csv(f, low_memory=False)
    d.columns = [c.strip() for c in d.columns]        # remove spaces from column names
    frames.append(d)
raw = pd.concat(frames, ignore_index=True)

print(f"Raw data: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

## 3. Feature extraction: 79 columns → 20 features
The table shows how original column names were converted to normalised feature names.

In [ ]:
missing = [k for k in RAW_MAP if k not in raw.columns]
assert not missing, f"Missing columns: {missing}"

pd.DataFrame({"original_column": list(RAW_MAP.keys()),
              "feature_name": list(RAW_MAP.values())})

In [ ]:
df = pd.DataFrame(index=raw.index)
for raw_col, feat in RAW_MAP.items():
    df[feat] = pd.to_numeric(raw[raw_col], errors="coerce")

# CICIDS2017 CSVs often have no 'Source Port' column; train_model.py then mirrors destination_port
if "Source Port" in raw.columns:
    df["source_port"] = pd.to_numeric(raw["Source Port"], errors="coerce")
    print("source_port taken from the raw 'Source Port' column")
else:
    df["source_port"] = df["destination_port"]
    print("No 'Source Port' column in raw data -> source_port mirrors destination_port")

df = df[FEATURE_COLS].copy()
print("Feature matrix shape:", df.shape)
df.head()

## 4. Target label
`is_attack` = 0 for `BENIGN`, 1 for any attack type. IP addresses and protocol names are not among the 20 features, so no one-hot encoding is needed; every feature is numeric.

In [ ]:
labels = raw["Label"].astype(str).str.strip()
df["label"] = labels
df["is_attack"] = (labels.str.upper() != "BENIGN").astype(int)

print(df["is_attack"].value_counts().rename({0: "BENIGN (0)", 1: "ATTACK (1)"}))
print()
print(df["label"].value_counts())

## 5. Remove NaN and Infinity values
These appear when a flow has a duration of zero (division by zero in the bytes/s and packets/s features).

In [ ]:
feat = df[FEATURE_COLS]
issues = pd.DataFrame({"inf_values": np.isinf(feat).sum(), "nan_values": feat.isna().sum()})
print("Columns containing NaN/Infinity:")
print(issues[(issues.inf_values > 0) | (issues.nan_values > 0)])

before = len(df)
df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
print(f"\nDropped {before - len(df):,} rows. Remaining: {len(df):,}")

if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"Sampled {SAMPLE_FRAC:.0%} of rows for a quick run: {len(df):,}")

## 6. Train / validation / test split
The scaler and the models are fitted on **benign traffic only**. Benign rows are split 80% train / 20% held-out, and the held-out part is split again into validation and test. Attack rows are never used for training; they are split evenly into validation and test, so the normal:attack ratio is the same in both sets.

In [ ]:
normal = df[df.is_attack == 0].sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
attack = df[df.is_attack == 1].sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

n_train = int(0.8 * len(normal))
train_normal = normal.iloc[:n_train]
held_normal = normal.iloc[n_train:]
half_n = len(held_normal) // 2
val_normal, test_normal = held_normal.iloc[:half_n], held_normal.iloc[half_n:]

half_a = len(attack) // 2
val_attack, test_attack = attack.iloc[:half_a], attack.iloc[half_a:]

pd.DataFrame({
    "set": ["train (benign only)", "validation", "test"],
    "normal_rows": [len(train_normal), len(val_normal), len(test_normal)],
    "attack_rows": [0, len(val_attack), len(test_attack)],
})

## 7. Scale the 20 features and train the four Isolation Forests
**Screenshot this cell for Section 5.4.1.** It shows the 20 features being scaled with `StandardScaler` and the four models being trained and saved.

In [ ]:
X_train = train_normal[FEATURE_COLS].values

# Scale the 20 features (fitted on benign traffic only)
scaler = StandardScaler().fit(X_train)
X_scaled = scaler.transform(X_train)
print("X_scaled shape:", X_scaled.shape, "-> 20 features")

dump(scaler, MODEL_DIR / "scaler.joblib")
with open(MODEL_DIR / "feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f)

# Train one Isolation Forest (200 trees) per contamination value
models = {}
for c in CONTAMINATIONS:
    t0 = time.time()
    model = IsolationForest(n_estimators=200, contamination=c, random_state=42, n_jobs=-1)
    model.fit(X_scaled)
    path = MODEL_DIR / f"if_contam_{str(c).replace('.', '_')}.joblib"
    dump(model, path)
    models[c] = model
    print(f"contamination={c:<4} trained in {time.time()-t0:5.1f}s -> {path.name}")

### Effect of scaling
A feature with huge values (`flow_bytes_per_s`) would dominate one with tiny values (`syn_flag_count`). After scaling both have mean 0 and standard deviation 1.

In [ ]:
cols = ["flow_bytes_per_s", "syn_flag_count"]
scaled_df = pd.DataFrame(X_scaled, columns=FEATURE_COLS)
pd.DataFrame({
    "mean_before": train_normal[cols].mean(),
    "std_before":  train_normal[cols].std(),
    "mean_after":  scaled_df[cols].mean(),
    "std_after":   scaled_df[cols].std(),
}).round(4)

## 8. Evaluation on unseen attack data
Standard k-fold cross-validation is not suitable here: the model is trained on one class (benign) only, so there are no attack labels to stratify folds with. Instead, unseen attack data is injected into the test set and the **decision threshold is swept**:

1. Score the validation set and pick the threshold that maximises F1.
2. Apply that threshold to the untouched test set and report ROC-AUC, Precision, Recall and F1.

Lower `decision_function` scores mean more anomalous, so a flow is flagged when its score is below the threshold.

In [ ]:
def score(model, d):
    return model.decision_function(scaler.transform(d[FEATURE_COLS].values))

def prf(y, pred):
    tp = (pred & y).sum(); fp = (pred & ~y).sum(); fn = (~pred & y).sum()
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    return p, r, 2 * p * r / max(p + r, 1e-12)

def sweep(s, y, n=200):
    thrs = np.quantile(s, np.linspace(0.001, 0.999, n))
    rows = [(t, *prf(y, s < t)) for t in thrs]
    return pd.DataFrame(rows, columns=["threshold", "precision", "recall", "f1"])

rows, curves, sweeps = [], {}, {}
for c, model in models.items():
    sv = np.concatenate([score(model, val_normal), score(model, val_attack)])
    yv = np.r_[np.zeros(len(val_normal), bool), np.ones(len(val_attack), bool)]
    thr = sweep(sv, yv).sort_values("f1").iloc[-1]["threshold"]      # best F1 on validation

    st = np.concatenate([score(model, test_normal), score(model, test_attack)])
    yt = np.r_[np.zeros(len(test_normal), bool), np.ones(len(test_attack), bool)]
    p, r, f = prf(yt, st < thr)
    auc = roc_auc_score(yt, -st)
    fpr_normal = float((st[~yt] < thr).mean())

    rows.append({"contamination": c, "threshold": round(thr, 4), "precision": p, "recall": r,
                 "f1": f, "roc_auc": auc, "false_positive_rate": fpr_normal})
    curves[c] = roc_curve(yt, -st)
    sweeps[c] = sweep(st, yt)

results = pd.DataFrame(rows).round(3)
results

### ROC curves for the four models

In [ ]:
plt.figure(figsize=(6, 5))
for c, (fpr, tpr, _) in curves.items():
    auc = results.loc[results.contamination == c, "roc_auc"].iloc[0]
    plt.plot(fpr, tpr, label=f"contamination={c} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="random guess")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves: Isolation Forest"); plt.legend(loc="lower right"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Precision, Recall and F1 across decision thresholds (best model)

In [ ]:
best_c = results.sort_values("f1").iloc[-1]["contamination"]
sw = sweeps[best_c]
plt.figure(figsize=(7, 4.5))
plt.plot(sw.threshold, sw.precision, label="Precision")
plt.plot(sw.threshold, sw.recall, label="Recall")
plt.plot(sw.threshold, sw.f1, label="F1-score")
plt.xlabel("Decision threshold (flag if score < threshold)"); plt.ylabel("Score")
plt.title(f"Threshold sweep, contamination={best_c}"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

results.to_csv(MODEL_DIR / "notebook_metrics.csv", index=False)
print("Saved metrics to", MODEL_DIR / "notebook_metrics.csv")